In [36]:
import sys
from pathlib import Path
import importlib
import pandas as pd

sys.path.append(str(Path.cwd().parent))

import data.loader as loader
import data.calculations as calc
import data.data_inventory as inv
import data.allocation as alloc

importlib.reload(loader)
importlib.reload(calc)
importlib.reload(inv)
importlib.reload(alloc)

# -----------------------------
# Load raw data
# -----------------------------
access_file = "/Users/ricardolugo/Library/CloudStorage/OneDrive-Personal/LH/Reports/sales_lh.accdb"
tables = ["sales", "customers"]

data = loader.load_tables(access_file, tables)

df_sales = data["sales"]
df_customers = data["customers"]
df_actividades, df_clasificacion, df_inventory = loader.load_excel_files()

# -----------------------------
# Build reference tables
# -----------------------------
df_grupos = calc.build_product_classification(df_clasificacion)
df_familias = calc.build_product_families(df_clasificacion)
df_customer_activity = calc.build_customer_activity(df_actividades)

# -----------------------------
# Enrich tables
# -----------------------------
df_sales_enriched = calc.enrich_sales_with_classification(
    df_sales,
    df_grupos,
    df_familias
)

df_customers_enriched = calc.enrich_customers_with_activity(
    df_customers,
    df_customer_activity
)

df_sales_final = calc.enrich_sales_with_customer_activity(
    df_sales_enriched,
    df_customers_enriched
)

# -----------------------------
# Inventory + model
# -----------------------------
df_inv = inv.prepare_inventory(df_inventory)

df_model = alloc.build_model(df_inv, df_sales_final)

df_alloc_result, df_low_rotation = alloc.run_allocation_simple(
    df_model,
    demand_threshold_cali=0.5,
    months_target_cali=2.0,
    months_reserve_bogota=3.0,
    min_post_bogota_units=2
)

df_dead_cali = alloc.get_dead_inventory_cali(df_model, df_sales_final)

Loading sales...
sales loaded: (285619, 25)
Loading customers...
customers loaded: (36429, 16)
Actividades shape: (594, 6)
Clasificacion shape: (203, 4)
Inventario shape: (11169, 32)


/Users/ricardolugo/Documents/dev/LH/project_cali/data/calculations.py:156: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["fecha"] = pd.to_datetime(
/Users/ricardolugo/Documents/dev/LH/project_cali/data/calculations.py:156: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["fecha"] = pd.to_datetime(


In [ ]:
print("SKUs sugeridos:", len(df_alloc_result))
print("Unidades sugeridas:", df_alloc_result["envio_sugerido"].sum())
print("Valor sugerido:", df_alloc_result["valor_envio"].sum())

SKUs sugeridos: 18
Unidades sugeridas: 54.0
Valor sugerido: 4949793.53


In [37]:
df_alloc_result[[
    "sku",
    "nombre",
    "familia",
    "stock_bogota",
    "stock_cali",
    "demanda_mensual_cali",
    "demanda_mensual_bogota",
    "stock_objetivo_cali",
    "necesidad_cali",
    "stock_reserva_bogota",
    "inventario_transferible_bogota",
    "envio_sugerido",
    "valor_envio"
]].sort_values("envio_sugerido", ascending=False).head(30)

,sku,nombre,familia,stock_bogota,stock_cali,demanda_mensual_cali,demanda_mensual_bogota,stock_objetivo_cali,necesidad_cali,stock_reserva_bogota,inventario_transferible_bogota,envio_sugerido,valor_envio
3126,PHC 08B-1X10FTSKF,CADENA SENCILLA,5000,17,1,3.683333,1.066667,8.0,7.0,4.0,13.0,7.0,970068.610
3035,PCM 121420 ESKF,BUJE PERMAGLIDE,1000,40,0,2.266667,0.000000,5.0,5.0,2.0,38.0,5.0,18689.575
4828,SIKB 10 FSKF,CABEZA DE ARTICULACION,1000,6,0,3.433333,0.000000,7.0,7.0,2.0,4.0,4.0,410720.000
66,12X22X7 HMSA10 RGSKF,RETENEDOR DE ACEITE,4000,8,0,1.133333,0.000000,3.0,3.0,2.0,6.0,3.0,13467.450
128,15X30X7 HMSA10 RGSKF,RETENEDOR DE ACEITE,4000,8,0,1.133333,0.000000,3.0,3.0,2.0,6.0,3.0,14085.870
1674,6304-2RSH/C3GJNSKF,RODAMIENTO RIGIDO DE BOLAS,1000,279,1,1.800000,1.933333,4.0,3.0,6.0,273.0,3.0,46826.385
338,22211 EKSKF,RODAMIENTO DE RODILLOS ESFERICOS,1000,6,1,1.416667,0.000000,3.0,2.0,2.0,4.0,2.0,821986.000
1611,625-2Z/C3SKF,RODAMIENTO RIGIDO DE BOLAS,1000,8,0,0.850000,0.000000,2.0,2.0,2.0,6.0,2.0,22017.500
2674,LGMT 3/1SKF,GRASA USO GERNERAL RODAMIENTOS,6000,8,0,0.566667,0.000000,2.0,2.0,2.0,6.0,2.0,82282.430
3673,PHG 5VX850EPSKF,CORREA TRAPEZOIDAL,5000,8,0,0.600000,0.000000,2.0,2.0,2.0,6.0,2.0,138967.370


In [38]:
df_alloc_result[df_alloc_result["familia"] == 1000].shape

(5, 27)

In [39]:
df_alloc_result.sort_values("envio_sugerido", ascending=False).to_excel(
    "traslado_cali_simple.xlsx",
    index=False
)